<a href="https://colab.research.google.com/github/thisishasan/speech_processing/blob/main/01_generate_questions_answers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip uninstall -y torchaudio

In [2]:
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "transformers", "accelerate", "bitsandbytes", "datasets", "pillow",
])

import json
import os
import re
import unicodedata
from collections import Counter, defaultdict
from io import BytesIO

import torch
from datasets import load_dataset
from google.colab import drive
from PIL import Image
from transformers import AutoProcessor, BitsAndBytesConfig
from transformers import Qwen3VLForConditionalGeneration

DRIVE_ROOT = "/content/drive/MyDrive/01_speech_processing/sp_exam_project/dataset"
DATASET_CACHE = f"{DRIVE_ROOT}/hf_cache"
IMAGE_DIR = f"{DRIVE_ROOT}/images"
OUTPUT_DIR = f"{DRIVE_ROOT}/outputs"
MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"

MAX_IMAGES = 3000
MAX_QA_PER_IMAGE = 5
USE_4BIT = True
SAVE_EVERY = 10
CHECKPOINT_FILE = os.path.join(OUTPUT_DIR, "checkpoint.json")

drive.mount("/content/drive")
for path in (DRIVE_ROOT, DATASET_CACHE, IMAGE_DIR, OUTPUT_DIR):
    os.makedirs(path, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > GPU in Colab.")

PARQUET_FILES = [
    "https://huggingface.co/datasets/nlphuji/flickr30k/resolve/"
    f"refs%2Fconvert%2Fparquet/TEST/test/{part:04d}.parquet"
    for part in range(9)
]

print("Loading Flickr30k from Parquet...")
flickr = load_dataset(
    "parquet",
    data_files={"flickr30k": PARQUET_FILES},
    split="flickr30k",
    cache_dir=DATASET_CACHE,
)
print(f"Dataset records: {len(flickr):,}")

if USE_4BIT:
    quantization = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    try:
        model = Qwen3VLForConditionalGeneration.from_pretrained(
            MODEL_ID, quantization_config=quantization, device_map="auto"
        )
    except (ImportError, RuntimeError) as error:
        print("4-bit loading is unavailable in this Colab environment.")
        print(f"Falling back to FP16: {error}")
        USE_4BIT = False

if not USE_4BIT:
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID, dtype=torch.float16, device_map="auto"
    )
processor = AutoProcessor.from_pretrained(MODEL_ID)

def clean(value):
    return re.sub(r"\s+", " ", str(value)).strip()


def normalize(value):
    value = unicodedata.normalize("NFKC", clean(value).lower())
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9\s]", " ", value)).strip()


def get_captions(record):
    captions = record.get("caption", record.get("raw", []))
    if isinstance(captions, str):
        try:
            captions = json.loads(captions)
        except json.JSONDecodeError:
            captions = [captions]
    return [clean(x) for x in captions if clean(x)]


def save_image(value, path):
    if os.path.exists(path):
        return
    if isinstance(value, Image.Image):
        image = value
    elif isinstance(value, dict) and value.get("bytes"):
        image = Image.open(BytesIO(value["bytes"]))
    elif isinstance(value, dict) and value.get("path"):
        image = Image.open(value["path"])
    else:
        raise TypeError(f"Unsupported image type: {type(value)}")
    image.convert("RGB").save(path, format="JPEG", quality=95)


def parse_json(text):
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)
    start, end = text.find("["), text.rfind("]")
    if start < 0 or end <= start:
        return []
    try:
        result = json.loads(text[start:end + 1])
        return result if isinstance(result, list) else []
    except json.JSONDecodeError:
        return []


def caption_support(answer, captions, support):
    if not isinstance(support, list):
        support = [support]
    indices = [int(x) for x in support if str(x).isdigit()]
    indices = [x for x in indices if 1 <= x <= len(captions)]
    if not indices:
        return False, []

    answer_words = set(normalize(answer).split())
    if not answer_words:
        return False, []
    best = 0.0
    for index in indices:
        caption_words = set(normalize(captions[index - 1]).split())
        best = max(best, len(answer_words & caption_words) / len(answer_words))
    return best >= 0.5, sorted(set(indices))


def generate_qas(image_path, captions):
    caption_block = "\n".join(
        f"{i}. {caption}" for i, caption in enumerate(captions, 1)
    )
    prompt = f"""
Look carefully at the image and read all five captions.
Generate up to {MAX_QA_PER_IMAGE} diverse visual question-answer pairs.

Rules:
- Every answer must be visible in the image.
- Every answer must be supported by at least one caption.
- Do not guess or use outside knowledge.
- Avoid duplicate questions.
- Prefer questions about actions, objects, people, colors, numbers, and scenes.
- Keep answers concise, normally 1 to 12 words.
- supporting_caption must contain caption numbers from 1 to 5.
- Return only a valid JSON array, without markdown.

Captions:
{caption_block}

Return exactly this structure:
[
  {{"question": "What is the person doing?", "answer": "...", "supporting_caption": [1]}}
]
""".strip()

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image_path},
            {"type": "text", "text": prompt},
        ],
    }]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=512, do_sample=False)
    generated = output[:, inputs.input_ids.shape[1]:]
    response = processor.batch_decode(
        generated, skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0]

    records, seen = [], set()
    for item in parse_json(response):
        if not isinstance(item, dict):
            continue
        question, answer = clean(item.get("question", "")), clean(item.get("answer", ""))
        supported, indices = caption_support(answer, captions, item.get("supporting_caption", []))
        key = (question.lower(), answer.lower())
        if not question or not answer or not supported or key in seen:
            continue
        records.append({
            "question": question,
            "answer": answer,
            "supporting_caption": indices,
            "qa_type": "multimodal_caption_grounded",
        })
        seen.add(key)
        if len(records) >= MAX_QA_PER_IMAGE:
            break
    return records

def save_outputs(records, failures, image_items, processed_filenames):
    """Write resumable progress to Drive."""
    by_split = defaultdict(list)
    for record in records:
        by_split[record["split"]].append(record)

    for split in ("train", "val", "test"):
        with open(os.path.join(OUTPUT_DIR, f"vqain_{split}.json"), "w", encoding="utf-8") as file:
            json.dump(by_split[split], file, ensure_ascii=False, indent=2)
        with open(os.path.join(OUTPUT_DIR, f"vqain_{split}.jsonl"), "w", encoding="utf-8") as file:
            for record in by_split[split]:
                file.write(json.dumps(record, ensure_ascii=False) + "\n")

    checkpoint = {
        "model": MODEL_ID,
        "completed": False,
        "images_processed": len(processed_filenames),
        "processed_filenames": sorted(processed_filenames),
        "qa_records": records,
        "failures": failures,
    }
    temporary = CHECKPOINT_FILE + ".tmp"
    with open(temporary, "w", encoding="utf-8") as file:
        json.dump(checkpoint, file, ensure_ascii=False)
    os.replace(temporary, CHECKPOINT_FILE)

images = {}
for row in flickr:
    filename = str(row["filename"])
    if filename in images:
        continue
    if MAX_IMAGES is not None and len(images) >= MAX_IMAGES:
        break
    split = str(row.get("split", "train")).lower()
    if split not in {"train", "val", "test"}:
        split = "train"
    path = os.path.join(IMAGE_DIR, filename)
    save_image(row["image"], path)
    images[filename] = {
        "filename": filename,
        "img_id": str(row.get("img_id", "")),
        "split": split,
        "path": path,
        "captions": get_captions(row),
    }

all_records, failures, processed_filenames = [], [], set()
if os.path.exists(CHECKPOINT_FILE):
    print(f"Resuming from checkpoint: {CHECKPOINT_FILE}")
    with open(CHECKPOINT_FILE, "r", encoding="utf-8") as file:
        checkpoint = json.load(file)
    all_records = checkpoint.get("qa_records", [])
    failures = checkpoint.get("failures", [])
    processed_filenames = set(checkpoint.get("processed_filenames", []))
    print(f"Already completed: {len(processed_filenames):,} images")

new_images = 0
for number, item in enumerate(images.values(), 1):
    if item["filename"] in processed_filenames:
        continue
    try:
        pairs = generate_qas(item["path"], item["captions"])
        for qa_number, pair in enumerate(pairs):
            all_records.append({
                "id": f"{item['split']}_{item['img_id']}_{number}_{qa_number}",
                "image": item["path"],
                "filename": item["filename"],
                "img_id": item["img_id"],
                "question": pair["question"],
                "answer": pair["answer"],
                "supporting_caption": pair["supporting_caption"],
                "captions": item["captions"],
                "qa_type": pair["qa_type"],
                "split": item["split"],
            })
    except Exception as error:
        failures.append({"filename": item["filename"], "error": str(error)})
    processed_filenames.add(item["filename"])
    new_images += 1
    if new_images % SAVE_EVERY == 0:
        save_outputs(all_records, failures, images, processed_filenames)
        print(f"Checkpoint saved: {len(processed_filenames):,} images; generated {len(all_records):,} QA records")

if new_images % SAVE_EVERY != 0 or not os.path.exists(CHECKPOINT_FILE):
    save_outputs(all_records, failures, images, processed_filenames)


summary = {
    "model": MODEL_ID,
    "images_processed": len(processed_filenames),
    "qa_records": len(all_records),
    "images_by_split": dict(Counter(item["split"] for item in images.values())),
    "qa_records_by_split": dict(Counter(record["split"] for record in all_records)),
    "failures": failures,
}
with open(os.path.join(OUTPUT_DIR, "summary.json"), "w", encoding="utf-8") as file:
    json.dump(summary, file, ensure_ascii=False, indent=2)

with open(CHECKPOINT_FILE, "w", encoding="utf-8") as file:
    checkpoint = {
        "model": MODEL_ID,
        "completed": True,
        "images_processed": len(processed_filenames),
        "processed_filenames": sorted(processed_filenames),
        "qa_records": all_records,
        "failures": failures,
    }
    json.dump(checkpoint, file, ensure_ascii=False)

print("\nCompleted:")
print(json.dumps(summary, indent=2))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading Flickr30k from Parquet...
Dataset records: 31,014


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

Resuming from checkpoint: /content/drive/MyDrive/01_speech_processing/sp_exam_project/dataset/outputs/checkpoint.json
Already completed: 2,230 images
Checkpoint saved: 2,240 images; generated 7,455 QA records
Checkpoint saved: 2,250 images; generated 7,488 QA records
Checkpoint saved: 2,260 images; generated 7,525 QA records
Checkpoint saved: 2,270 images; generated 7,559 QA records
Checkpoint saved: 2,280 images; generated 7,594 QA records
Checkpoint saved: 2,290 images; generated 7,629 QA records
Checkpoint saved: 2,300 images; generated 7,660 QA records
Checkpoint saved: 2,310 images; generated 7,699 QA records
Checkpoint saved: 2,320 images; generated 7,733 QA records
Checkpoint saved: 2,330 images; generated 7,762 QA records
Checkpoint saved: 2,340 images; generated 7,802 QA records
Checkpoint saved: 2,350 images; generated 7,836 QA records
Checkpoint saved: 2,360 images; generated 7,876 QA records
Checkpoint saved: 2,370 images; generated 7,906 QA records
Checkpoint saved: 2,380 